# Compare task configs: baselines vs foundation models

This notebook compares fold-related (and lag grid) settings across three locations:

1. **Saved runs** — `baseline-results/<experiment>/` — looks for `configs.yaml`, `config.yaml`, then `config.yml` (this repo uses flattened `config.yml` with top-level `task_name` / `training_params`).
2. **Neural conv templates** — `configs/baselines/neural_conv_decoder/*.yml` — nested `task_config.task_name` + `training_params`.
3. **Foundation templates** — `configs/foundation_models/<model>/<task_dir>/` — loads a representative YAML (`supersubject.yml` preferred, else first `*.yml` / `*.yaml`).

**Note:** template file stems (e.g. `arbitrary.yml`) may differ from foundation `task_dir` (e.g. `word_embedding/`). Rows are aligned by YAML `task_name`, with stems/dirs kept for traceability.

`scan_foundation_models(..., skip_model_dirs=...)` skips `legacy/` by default; edit that call in the code cell if you want it included.

Set `REPO_ROOT` in the first code cell if your cwd is not the repository root.

**Performance:** the second code cell logs timing for each scan. `baseline-results` can be slow on parallel filesystems; set `SCAN_BASELINE_RESULTS = False` there to skip it, or set `BASELINE_RESULTS_MAX_EXPERIMENTS` to cap how many run folders are read.

In [ ]:
BASE_PATH = "/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/"
BASELINE_RESULT_PATH = BASE_PATH + "baseline-results/"

BASELINE_TASK_MODEL_COMBINATION_LIST = [
    "content_noncontent_task_sig_elecs_mlp_early_stop_roc_2025-12-19-00-34-17",
    "llm_decoding_control_2025-12-28-15-55-38",
    "ensemble_model_10_arbitrary_2025-12-19-00-17-32",
    "llm_token_finetune_2025-12-26-12-44-36",
    "ensemble_model_10_glove_2025-12-19-00-17-41",
    "neural_conv_whisper_embedding_2026-02-17-19-25-13",
    "ensemble_model_10_gpt2_2026-02-17-19-54-55",
    "pos_task_sig_elecs_without_other_classes_2025-12-19-00-34-17",
    "gpt_surprise_2025-12-19-00-18-43",
    "sentence_onset_lr_2025-12-19-00-18-44",
    "gpt_surprise_2025-12-19-00-18-44",
    "volume_level_simple_2025-12-19-00-34-56",
    "iu_boundary_lr_2026-02-26-09-46-13",
    "volume_level_torch_ridge_2025-12-19-00-42-42"
]
# add logic : for ensemble_model


TASK = #select task

In [18]:
CONFIGS_PATH = "/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs"
BASELINE_PATH = "/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/baselines/neural_conv_decoder"

# task_list = [
#     "content_noncontent.yml",
#     "gpt_surprise_multiclass.yml",
#     "llm_decoding.yml",
#     "pos.yml",
#     "volume_level.yml",
#     "word_embedding.yml",
#     "gpt_surprise.yml",
#     "iu_boundary.yml",
#     "llm_embedding_pretraining.yml",
#     "sentence_onset.yml",
#     "whisper_embedding.yml"
# ]

BRAINBERT_PATH = "/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/"
task_list = [
    "content_noncontent",
    "gpt_surprise_multiclass",
    "llm_decoding",
    "pos",
    "volume_level",
    "word_embedding",
    "gpt_surprise",
    "iu_boundary",
    "llm_embedding_pretraining",
    "sentence_onset",
    "whisper_embedding",
]

task_idx = 3 #0 to 10
import os

# Get the first task name from task_list
task_name = task_list[task_idx]

# Build the config file paths as strings
config_path = os.path.join(BASELINE_PATH, f"{task_name}.yml")
brainbert_config_path = os.path.join(BRAINBERT_PATH, task_name, "supersubject.yml")

with open(config_path, 'r') as f:
    print(f"Content of {config_path}:\n")
    print(f.read())

Content of /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/baselines/neural_conv_decoder/pos.yml:

config_setter_name: neural_conv
task_config:
  task_name: pos_task
  data_params:
    data_root: data
    electrode_file_path: processed_data/all_subject_sig.csv
    window_width: 0.625
    preprocessing_fn_name: window_average_neural_data
    preprocessor_params:
      num_average_samples: 8
  task_specific_config:
    pos_path: processed_data/df_word_onset_with_pos_class.csv
training_params:
  batch_size: 32
  epochs: 100
  learning_rate: 0.001
  weight_decay: 0.0001
  # Learning rate scheduler: reduces LR when validation metric plateaus
  use_lr_scheduler: true
  scheduler_params:
    factor: 0.5    # Reduce LR by half when metric plateaus
    patience: 10   # Wait 10 epochs without improvement
    min_lr: 1e-6   # Minimum learning rate
    # mode is automatically set to "max" because smaller_is_better: false
  early_stopping_patience: 100
  n_folds: 5
  min_lag: -100

In [19]:
with open(brainbert_config_path, 'r') as f:
    print(f"\nContent of {brainbert_config_path}:\n")
    print(f.read())


Content of /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/pos/supersubject.yml:

config_setter_name: brainbert_finetune
task_config:
  task_name: pos_task
  data_params:
    data_root: data
    electrode_file_path: processed_data/all_subject_sig.csv
    use_high_gamma: false
    target_sr: 2048
    window_width: 1.0
  task_specific_config:
    pos_path: processed_data/df_word_onset_with_pos_class.csv
training_params:
  batch_size: 4
  epochs: 100
  learning_rate: 1.0e-05
  weight_decay: 1.0e-05
  early_stopping_patience: 10
  n_folds: 5
  min_lag: -1000
  max_lag: 1500
  lag_step_size: 500
  losses:
  - cross_entropy
  loss_weights:
  - 1.0
  metrics:
  - roc_auc_multiclass
  - f1
  - acc
  early_stopping_metric: roc_auc_multiclass
  smaller_is_better: false
  fold_ids:
  - 1
  - 5
trial_name: brainbert_supersubject_pos
model_spec:
  constructor_name: brainbert_finetune
  params:
    model_dir: models/brainbert/pretrained_model
    output

In [ ]:
# Set training_params.n_folds: only YAMLs with n_folds==SOURCE_N_FOLDS are rewritten to TARGET_N_FOLDS.
# (e.g. diver/content_noncontent/subject1_full.yml — files live as task/subject1_full.yml, not task/subject1/_full.yml)
from pathlib import Path

import yaml

ROOT = Path("/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models")
MODELS = ["brainbert", "diver", "popt"]
TASKS = ["content_noncontent", "gpt_surprise_multiclass", "gpt_surprise"]
SUBJECT_TAGS = [f"subject{i}" for i in range(1, 10)]
TARGET_N_FOLDS = 5
DRY_RUN = False  # True: print paths that would change, do not write


def candidate_paths(base: Path):
    names = [
        "supersubject.yml",
        "persubject_concat.yml",
    ]
    for s in SUBJECT_TAGS:
        names.extend([f"{s}_full.yml"])
    return [base / n for n in names]


def classify_and_patch(p: Path) -> str:
    """missing_file | missing_task_dir outside | already_target | skipped_other | no_n_folds | not_dict | would_change | changed"""
    if not p.is_file():
        return "missing_file"
    doc = yaml.safe_load(p.read_text(encoding="utf-8"))
    if not isinstance(doc, dict):
        return "not_dict"
    tp = doc.get("training_params")
    if not isinstance(tp, dict) or "n_folds" not in tp:
        return "no_n_folds"
    nf = tp.get("n_folds")
    if nf == TARGET_N_FOLDS:
        return "already_target"
    if nf != SOURCE_N_FOLDS:
        return "skipped_other"
    tp["n_folds"] = TARGET_N_FOLDS
    if DRY_RUN:
        print(f"[dry-run] would set n_folds {SOURCE_N_FOLDS}->{TARGET_N_FOLDS}: {p}")
        return "would_change"
    out = yaml.dump(
        doc,
        default_flow_style=False,
        sort_keys=False,
        allow_unicode=True,
    )
    p.write_text(out, encoding="utf-8")
    print(p)
    return "changed"


counts = {
    "missing_file": 0,
    "already_target": 0,
    "skipped_other": 0,
    "no_n_folds": 0,
    "not_dict": 0,
    "would_change": 0,
    "changed": 0,
}
missing_task_dir = 0

for model in MODELS:
    for task in TASKS:
        base = ROOT / model / task
        if not base.is_dir():
            missing_task_dir += 1
            continue
        for p in candidate_paths(base):
            counts[classify_and_patch(p)] += 1

candidates = sum(counts.values())
print(
    f"\n--- summary (per candidate path under existing task dirs) ---\n"
    f"  missing task directory (model/task not found): {missing_task_dir}\n"
    f"  missing file (candidate path not on disk):     {counts['missing_file']}\n"
    f"  already n_folds={TARGET_N_FOLDS} (unchanged):     {counts['already_target']}\n"
    f"  n_folds present but not {SOURCE_N_FOLDS} (skip):  {counts['skipped_other']}\n"
    f"  no training_params.n_folds:                     {counts['no_n_folds']}\n"
    f"  top-level not a dict:                          {counts['not_dict']}\n"
    f"  {'would change (dry-run)' if DRY_RUN else 'written'}:                        {counts['would_change'] + counts['changed']}\n"
    f"  (checked candidate paths):                     {candidates}\n"
)

/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/content_noncontent/supersubject.yml
/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/content_noncontent/persubject_concat.yml
/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/content_noncontent/subject1_full.yml
/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/content_noncontent/subject2_full.yml
/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/content_noncontent/subject3_full.yml
/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/content_noncontent/subject4_full.yml
/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_models/brainbert/content_noncontent/subject5_full.yml
/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/configs/foundation_mod